# Week 18 -- RLHF & PPO for Language Models -- Workbook

This workbook has two explicit exercise modes, controlled by `REFERENCE_MODE`
in the setup cell.

- **Reference (`True`, default):** supplied implementations demonstrate the
  calculations. Passing these checks verifies the reference, not your code.
- **Learner (`False`):** complete each scaffold's function, then restart and
  run all cells. The corresponding checks call your function. An unfinished
  scaffold raises `NotImplementedError`; there is no reference fallback.
  In a reference-wrapper scaffold, replace its guarded placeholder body with
  your implementation before choosing learner mode.

Change the setup flag before restarting. Record a boundary-case prediction
before each check and change an input after it passes. Later experiment cells
without a scaffold are reference demonstrations, not automated learner grades.

**Teaching companion:** [Worked examples, assumptions and self-checks](teaching_guide.md).
For additional independently graded tasks, see the
[progressive coding projects](../docs/INTERACTIVE_LEARNING.md).


**A bounded first investigation:** Trace one response through all four probability roles. Start with **Exercises 2, 3, 4 and 5** and the
[guide’s prediction, visual diagnostic and changed-input check](teaching_guide.md#notebook-investigation-trace-one-response-through-all-four-probability-roles).
Save your own prediction before running the reference cells. For your own implementation, choose learner mode in setup, restart, and verify
that the corresponding check calls your completed function.


**Worked examples and derivations:** [Teaching guide](teaching_guide.md) — prerequisite links, analogy boundaries, calculated examples, code connections, and retrieval questions.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import torch, torch.nn.functional as F, numpy as np
from project import solutions as S
from project.data import ToyPreferenceDataset, heuristic_reward, encode, decode
from project.model import CharTransformer, RewardModel, TransformerConfig
from project.sft import sft_train
from project.rm import rm_train
from project.ppo import compute_gae, ppo_train, PPOConfig
# True runs documented references. False checks YOUR completed functions.
# Restart and run all after changing this flag; an unfinished learner stops.
REFERENCE_MODE = True
def select_exercise(reference, learner):
    return reference if REFERENCE_MODE else learner
print("Exercise mode:", "reference demonstration" if REFERENCE_MODE else "learner checks (no fallback)")


## Exercise 1 (star) -- Bradley--Terry loss
Implement the pairwise log-likelihood loss: given two reward scores `r_w` and `r_l`, return $-\log\sigma(r_w - r_l)$ averaged over the batch.

> **Hint** (try yourself first): use `F.logsigmoid`, not `torch.log(torch.sigmoid(...))`.

In [ ]:
def bt_loss(r_w, r_l):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
torch.manual_seed(0)
rw = torch.randn(8); rl = torch.randn(8)
ref = select_exercise(S.exercise_1_bt_loss, bt_loss)(rw, rl)
assert torch.isfinite(ref).all()
assert ref.item() > 0
print('reference loss =', float(ref))
equal_loss = select_exercise(S.exercise_1_bt_loss, bt_loss)(torch.zeros(2), torch.zeros(2))
assert abs(float(equal_loss) - np.log(2)) < 1e-6
good_order = select_exercise(S.exercise_1_bt_loss, bt_loss)(torch.tensor([2.]), torch.tensor([0.]))
bad_order = select_exercise(S.exercise_1_bt_loss, bt_loss)(torch.tensor([0.]), torch.tensor([2.]))
assert float(good_order) < float(equal_loss) < float(bad_order)

# Independently calculate logistic NLL; shared reward shifts cancel.
expected_bt=np.logaddexp(0,(rl-rw).numpy()).mean()
assert abs(float(ref)-float(expected_bt))<1e-6
shifted=select_exercise(S.exercise_1_bt_loss, bt_loss)(rw+7,rl+7)
assert abs(float(shifted)-float(ref))<1e-6


## Exercise 2 (star) -- token-level KL between two categorical distributions

> **Hint**: KL(P || Q) = sum_v P(v) (log P(v) - log Q(v)).  Use the log-softmax for stability.

In [ ]:
def token_kl(logits_p, logits_q):
    raise NotImplementedError

In [ ]:
torch.manual_seed(1)
lp, lq = torch.randn(4, 7), torch.randn(4, 7)
ref = select_exercise(S.exercise_2_token_kl, token_kl)(lp, lq)
assert ref.shape == (4,)
assert (ref >= -1e-6).all()
print('reference KL per sample =', ref)
# KL((.5,.5)||(.75,.25))=.5*log(4/3), in nats.
fixture_kl=select_exercise(S.exercise_2_token_kl, token_kl)(
    torch.log(torch.tensor([[.5,.5]])),torch.log(torch.tensor([[.75,.25]])))
assert abs(float(fixture_kl)-.5*np.log(4/3))<1e-6


## Exercise 3 (star star) -- PPO clipped surrogate

> **Hint**: r_t = exp(log pi_theta - log pi_old).  Take the min of unclipped and clamped ratio times the advantage; mind the leading minus sign.

In [ ]:
def ppo_loss(new_logp, old_logp, adv, clip_eps=0.2):
    raise NotImplementedError

In [ ]:
torch.manual_seed(2)
new_logp = torch.randn(8, 4)
old_logp = new_logp + 0.05 * torch.randn(8, 4)
adv = torch.randn(8, 4)
ref = select_exercise(S.exercise_3_ppo_loss, ppo_loss)(new_logp, old_logp, adv, 0.2)
assert torch.isfinite(ref)
print('reference PPO loss =', float(ref))
# Positive ratio1.5 caps at1.2; negative ratio.5 uses -.8: loss=-(1.2-.8)/2.
fixture_ppo=select_exercise(S.exercise_3_ppo_loss, ppo_loss)(
    torch.log(torch.tensor([1.5,.5])),torch.zeros(2),torch.tensor([1.,-1.]),.2)
assert abs(float(fixture_ppo)+.2)<1e-6


## Exercise 4 (star star) -- GAE
Generalised Advantage Estimation: given rewards (B,T) and bootstrapped values (B,T+1), return advantages and returns.

> **Hint**: iterate t from T-1 down to 0, accumulating delta_t + gamma*lam*last.

In [ ]:
def my_gae(rewards, values, gamma=1.0, lam=0.95):
    raise NotImplementedError

In [ ]:
torch.manual_seed(3)
r = torch.randn(2, 5); v = torch.randn(2, 6)
adv_ref, ret_ref = select_exercise(S.exercise_4_gae, my_gae)(r, v)
assert adv_ref.shape == (2, 5) and ret_ref.shape == (2, 5)
print('adv[0]=', adv_ref[0])
fixture_adv,fixture_ret=select_exercise(S.exercise_4_gae, my_gae)(
    torch.tensor([[1.,1.]]),torch.zeros((1,3)),gamma=1.,lam=1.)
torch.testing.assert_close(fixture_adv,torch.tensor([[2.,1.]]))
torch.testing.assert_close(fixture_ret,torch.tensor([[2.,1.]]))


## Exercise 5 (star star) -- adaptive KL controller update

> **Hint**: beta <- beta * (1 + clip(observed/target - 1, -0.2, 0.2) / H).

In [ ]:
def kl_update(beta, observed, target, horizon):
    raise NotImplementedError

In [ ]:
beta_up   = select_exercise(S.exercise_5_kl_update, kl_update)(0.05, 0.20, 0.10, 10)
beta_down = select_exercise(S.exercise_5_kl_update, kl_update)(0.05, 0.05, 0.10, 10)
assert beta_up > 0.05 and beta_down < 0.05
print('beta up   =', beta_up)
print('beta down =', beta_down)

## Exercise 6 (star star) -- length-normalised reward
Penalise verbose responses by subtracting alpha * len(y) from the reward.

> **Hint**: a one-liner; cast lengths to float to keep gradients well-typed.

In [ ]:
def length_norm_reward(reward, length, alpha=0.01):
    raise NotImplementedError

In [ ]:
r = torch.tensor([1.0, 2.0]); L = torch.tensor([10, 80])
out = select_exercise(S.exercise_6_length_normalised, length_norm_reward)(r, L, alpha=0.02)
assert out[1] < out[0]
print('lengths penalised =>', out)
torch.testing.assert_close(out,torch.tensor([.8,.4]),atol=1e-6,rtol=0)


## Exercise 7 (star star star) -- closed-form RLHF optimum (DPO seed)
Return the unnormalised log probability log pi*(y) = log pi_ref(y) + r(y)/beta.

> **Hint**: one line, but understand that inverting it gives next week's DPO loss.

In [ ]:
def optimal_logp(log_pi_ref, reward, beta):
    raise NotImplementedError

In [ ]:
lp = torch.tensor([-1.0, -2.0]); r = torch.tensor([0.5, -0.5])
out = select_exercise(S.exercise_7_optimal_policy_logprob, optimal_logp)(lp, r, beta=0.1)
expected = torch.tensor([-1.0 + 5.0, -2.0 - 5.0])
assert torch.allclose(out, expected)
print('optimal logp =', out)

## Exercise 8 (star star star) -- *Open-ended:* proxy-vs-gold overoptimisation
Train SFT + RM + PPO on the toy dataset, then sample completions and compare the learned reward to the heuristic gold reward.  A large positive gap signals reward hacking.

> **Hint**: use `S.exercise_8_overoptimisation_score(learned, gold)`.  Push PPO steps higher and watch the gap grow.

In [ ]:
ds = ToyPreferenceDataset(n=32, seed=0)
cfg = TransformerConfig()
policy = CharTransformer(cfg)
rm = RewardModel(CharTransformer(cfg))
sft_train(policy, ds.sft_strings(), steps=60, seed=0)
rm_train(rm, ds.pairs, steps=60, seed=0)
hist, _, _ = ppo_train(policy, rm, ds.prompts(), cfg=PPOConfig(steps=8, batch_size=2, max_new_tokens=12))
prompt_ids = torch.tensor([encode(ds.prompts()[0])])
out = policy.generate(prompt_ids, max_new_tokens=12, greedy=False)
text = decode(out[0])
learned = rm(out, torch.tensor([out.size(1)])).detach()
gold = torch.tensor([heuristic_reward(text)])
gap = S.exercise_8_overoptimisation_score(learned, gold)
print('sample:', text)
print('learned:', float(learned), 'gold:', float(gold), 'gap:', gap)

## Exercise 9 (star star) -- *Meaningful objective:* heuristic-vs-learned correlation
Pearson correlation between the heuristic gold reward and the learned RM scores on a batch of generations.  A drop in correlation after PPO is a clean signal of reward hacking.

> **Hint**: `S.exercise_9_reward_correlation(texts, scores)` does the maths.

In [ ]:
texts, scores = [], []
for p in ds.prompts()[:8]:
    ids = torch.tensor([encode(p)])
    out = policy.generate(ids, max_new_tokens=12, greedy=False)
    text = decode(out[0])
    score = float(rm(out, torch.tensor([out.size(1)])).detach())
    texts.append(text); scores.append(score)
rho = S.exercise_9_reward_correlation(texts, scores)
print(f'Pearson rho (heuristic, learned) = {rho:.3f}')
assert -1.0 <= rho <= 1.0

## Wrap-up
If every cell above ran green, you have a working three-stage RLHF pipeline on a toy character-level model.  The TRL recipe in `project/trl_recipe.py` extends it to Pythia-160M or SmolLM2-360M on a single 24 GB GPU.

Next week we use the closed-form optimum from Exercise 7 to derive Direct Preference Optimization -- the same alignment story without PPO.

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 (star) -- Bradley--Terry loss</summary>

> **Hint** (try yourself first): use `F.logsigmoid`, not `torch.log(torch.sigmoid(...))`.

</details>

<details>
<summary>Exercise 2 (star) -- token-level KL between two categorical distributions</summary>

> **Hint**: KL(P || Q) = sum_v P(v) (log P(v) - log Q(v)).  Use the log-softmax for stability.

</details>

<details>
<summary>Exercise 3 (star star) -- PPO clipped surrogate</summary>

> **Hint**: r_t = exp(log pi_theta - log pi_old).  Take the min of unclipped and clamped ratio times the advantage; mind the leading minus sign.

</details>

<details>
<summary>Exercise 4 (star star) -- GAE</summary>

> **Hint**: iterate t from T-1 down to 0, accumulating delta_t + gamma*lam*last.

</details>

<details>
<summary>Exercise 5 (star star) -- adaptive KL controller update</summary>

> **Hint**: beta <- beta * (1 + clip(observed/target - 1, -0.2, 0.2) / H).

</details>

<details>
<summary>Exercise 6 (star star) -- length-normalised reward</summary>

> **Hint**: a one-liner; cast lengths to float to keep gradients well-typed.

</details>

<details>
<summary>Exercise 7 (star star star) -- closed-form RLHF optimum (DPO seed)</summary>

> **Hint**: one line, but understand that inverting it gives next week's DPO loss.

</details>

<details>
<summary>Exercise 8 (star star star) -- *Open-ended:* proxy-vs-gold overoptimisation</summary>

> **Hint**: use `S.exercise_8_overoptimisation_score(learned, gold)`.  Push PPO steps higher and watch the gap grow.

</details>

<details>
<summary>Exercise 9 (star star) -- *Meaningful objective:* heuristic-vs-learned correlation</summary>

> **Hint**: `S.exercise_9_reward_correlation(texts, scores)` does the maths.

</details>

After solving, change one input or assumption and predict the result before
running it. In reference mode, passing checks verifies the demonstration. In learner mode,
the selected checks must pass against your completed functions without fallback.


## Statistical investigation — Many completions from one prompt share evidence

**Goal.** Identify the remaining dependence after forming paired policy-score differences. This bounded reference investigation runs on CPU without downloads.
Write your prediction in a new Markdown cell before executing it.

**Predict.** A frozen policy pair is compared on 50 independent prompts, with six
stochastic completions per prompt. Shared prompt difficulty correlates the two policies’ scores. A shared
prompt-specific gain correlates the six completion-level differences within
each prompt; that is the clustering mechanism for the paired comparison.
Predict the difference between a completion-row interval and a paired
prompt-bootstrap interval. This fixture concerns evaluation design, not an
RLHF training outcome.

**Work a small case by hand.** For completion k of prompt j, the paired score difference is
D_jk=g_j+e_Bjk-e_Ajk. The shared difficulty cancels exactly. Here
Var(g)=0.4²=0.16 and each completion noise has variance 0.2²=0.04, so
Var(D_jk)=0.24 and Cov(D_jk,D_jl)=0.16 for k≠l. Averaging six completions
gives variance 0.16+0.08/6≈0.1733 per prompt. Across 50 prompts the
planning SE is sqrt(0.1733/50)≈0.0589; a completion-IID shortcut instead
uses sqrt(0.24/300)≈0.0283.

**Reading bridge:** [Probability and statistics lesson](../docs/statistics/04-intervals-comparisons.md).
These are original course examples informed by the textbook sections in the
[statistics source catalog](../docs/statistics/SOURCES.md); textbook PDFs are not required to run them.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from common.plotting import RL_PALETTE, use_rl_style
use_rl_style()
from common.statistics import paired_bootstrap_difference, independent_mean_interval
rng_stats=np.random.default_rng(118);n_prompts=50;completions=6
difficulty=rng_stats.normal(0,1,(n_prompts,1));prompt_gain=rng_stats.normal(.15,.4,(n_prompts,1))
score_a=difficulty+rng_stats.normal(0,.2,(n_prompts,completions))
score_b=difficulty+prompt_gain+rng_stats.normal(0,.2,(n_prompts,completions))
paired=paired_bootstrap_difference(score_b.mean(1),score_a.mean(1),seed=118,unit='independent prompt; B-A')
naive=independent_mean_interval((score_b-score_a).ravel(),unit='completion (incorrect independent unit)')
print('Paired prompt bootstrap:',paired)
print('Naive completion interval:',{k:naive[k] for k in ['estimate','low','high','n']})
fig,axes=plt.subplots(1,2,figsize=(9,3.3))
axes[0].scatter(score_a.mean(1),score_b.mean(1),s=20);axes[0].set(xlabel='Policy A prompt mean score',ylabel='Policy B prompt mean score')
axes[1].hist((score_b-score_a).mean(1),bins=15)
axes[1].axvline(0,color=RL_PALETTE['gray'],ls='--');axes[1].set(xlabel='Within-prompt policy difference B-A',ylabel='Independent prompts')
fig.tight_layout();plt.show()


**Interpret and transfer.** Completion replication can reduce noise within a prompt while leaving the
number of independent prompts unchanged. These intervals exclude prompt-set
selection, reward-model misspecification, and training-seed variation.

**Transfer.** Several prompts are paraphrases of the same user request. Define
the outer cluster, preserve both policies inside it, and explain why a narrow
reward-model interval says nothing by itself about human benefit.


**Read your result.** The diagonal relationship in policy scores reflects shared difficulty.
The histogram and interval concern paired differences, whose clustering
comes from prompt-specific gain. Compare the prompt-bootstrap width with
the invalid completion-IID width without attributing both to the same mechanism.

<details><summary>Worked feedback for the changed assumption</summary>

If several prompts are paraphrases of one request, preserve all their
policy-A and policy-B outcomes within a request cluster. Independent requests
then provide the outer units. The interval still concerns the scoring
endpoint under that population; it cannot establish that the reward model
measures human benefit or that the policy generalizes to new task types.

</details>
